In [ ]:
import os
from openai import OpenAI
from IPython.display import display,Markdown
from dotenv import load_dotenv

import requests

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print(f"OpenAI API key not found")

if google_api_key:
    print(f"Gemini API key exists and begins {google_api_key[:8]}")
else:
    print(f"Gemini API key not found")

In [ ]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
ollama = OpenAI(base_url=ollama_url,api_key='ollama')

In [ ]:
# GPT 5 nano = George, GPT 5 mini = Graham, Ollama = Ollie

george_system_prompt = """
You are George, a chatbot who is very argumentative.
You disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Graham and Ollie.
"""

graham_system_prompt = """
You are Graham, you are a very polite, courteous chatbot. You try to agree with everything the other person says,
or find common ground. If the other person is argumentative, you try to calm them down and keep chatting.
You are in a conversation with George and Ollie.
"""

ollie_system_prompt = """
    Your name is Ollie. You are a chaotic chatbot.
    You speak in short, philosophical and slightly confusing riddles.
    You act as the wildcard mediator between George and Graham.
"""

george_messages = ["Hi there"]
graham_messages = ["Hi George, glad to be here!"]
ollie_messages = ["Greetings, travellers of thought!"]


In [ ]:
george_model = "gpt-5-mini"
#graham_model = "gemini-2.5-flash-lite"
graham_model = "gpt-5-nano"
ollie_model = "llama3.2"

In [ ]:
def chat_george():
    messages = [{"role":"system","content":george_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"assistant","content":george})
        messages.append({"role":"user","content":f"Graham: {graham}\nOllie: {ollie}"})
    response = openai.chat.completions.create(model=george_model,messages=messages)
    return response.choices[0].message.content    

In [ ]:
def chat_graham():
    messages = [{"role":"system","content":graham_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"user","content":f"George: {george}\nOllie: {ollie}"})
        messages.append({"role":"assistant","content":graham})
    messages.append({"role":"user","content":f"George: {george_messages[-1]}\nOllie: {ollie_messages[-1]}"})
    #response = gemini.chat.completions.create(model=graham_model,messages=messages)
    response = openai.chat.completions.create(model=graham_model,messages=messages)
    return response.choices[0].message.content
    

In [ ]:
def chat_ollie():
    messages = [{"role":"system","content":ollie_system_prompt}]
    for george,graham,ollie in zip(george_messages,graham_messages,ollie_messages):
        messages.append({"role":"user","content":f"George: {george}\nGraham: {graham}"})
        messages.append({"role":"assistant","content":ollie})
    messages.append({"role":"user","content":f"George: {george_messages[-1]}\nGraham: {graham_messages[-1]}"})
    response = ollama.chat.completions.create(model=ollie_model,messages=messages)
    return response.choices[0].message.content

In [ ]:
# george_messages = ["Hi there"]
# graham_messages = ["Hi George, glad to be here!"]
# ollie_messages = ["Greetings, travellers of thought!"]

display(Markdown(f"### George:\n{george_messages[0]}\n"))
display(Markdown(f"### Graham:\n{graham_messages[0]}\n"))
display(Markdown(f"### Ollie:\n{ollie_messages[0]}\n"))

for i in range(2):
    print("-" * 50)
    display(Markdown(f"## Round {i+1}\n"))
    
    george_next = chat_george()
    display(Markdown(f"### George:\n{george_next}\n"))
    george_messages.append(george_next)    
    
    graham_next = chat_graham()
    display(Markdown(f"### Graham:\n{graham_next}\n"))
    graham_messages.append(graham_next)
    
    ollie_next = chat_ollie()
    display(Markdown(f"### Ollie:\n{ollie_next}\n"))
    ollie_messages.append(ollie_next)